# K-Nearest Neighbors (KNN)

**Author:** Saksham Chauhan

**Education**
- B.Sc. (Hons.) Statistics, University of Delhi
- M.Sc. Statistics, University of Delhi
- M.Tech. Artificial Intelligence & Data Science, IIIT Kottayam

**Connect**
- LinkedIn: https://linkedin.com/in/sakshamstats
- GitHub: https://github.com/sakshamchauhan-stats

# 1. Introduction

K-Nearest Neighbors (KNN) is a non-parametric, instance-based supervised machine learning algorithm used for both classification and regression. Instead of learning an explicit mathematical model during training, KNN stores the training data and predicts the output of a new observation based on the labels or target values of its nearest neighbors. The fundamental assumption behind KNN is: Similar observations are likely to have similar outcomes. \ 
The similarity between observations is typically measured using a distance metric such as Euclidean, Manhattan or Minkowski distance. \

When a new data point is presented, the algorithm finds the K closest training observations and uses their outputs to make a prediction.

- In **KNN Classification**, the new observation is assigned the class that appears most frequently among its K nearest neighbors.
- In **KNN Regression**, the predicted value is the average (or distance-weighted average) of the target values of the K nearest neighbors.

### Algorithm Family
- Supervised
- Non-Parametric
- Instance-Based/Lazy Learner
- Deterministic
- Discriminative
- Online (no training phase)

---

### Algorithm

#### Classification

1. Compute the distance between the new observation and every training sample.
2. Select the K nearest neighbors.
3. Count the frequency of each class.
4. Predict the majority class, or use distance weighted voting.


#### Regression

1. Compute the distance between the new observation and every training sample.
2. Select the K nearest neighbors.
3. Compute the average (or weighted average) of their target values.
4. Return the average as the prediction.

---

### When Does KNN Perform Well?

KNN generally performs well when

- The dataset is relatively small.
- Similar observations belong to similar classes or have similar target values.
- Features have been properly scaled.
- The number of features is moderate.
- Decision boundaries are non-linear.
- Noise and outliers are limited.


### When Should KNN be Avoided?

KNN is generally not recommended when

- The dataset is extremely large.
- The number of features is very high.
- Features are measured on different scales without normalization.
- There are many irrelevant features.
- The dataset contains significant noise or many outliers.

# 2. Scikit-Learn Implementation

###  Classification Dataset
The Breast Cancer Wisconsin dataset contains diagnostic measurements computed from digitized images of breast tissue samples. The objective is to classify each tumor as either Malignant (M) or Benign (B) based on these features.

### Regression Dataset
The California Housing Prices dataset contains demographic, geographic and housing-related information collected from California census blocks. The objective is to predict the median house value of a district using its socio-economic and geographical characteristics.

In [13]:
# Import required libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import kagglehub

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    StratifiedKFold,
    KFold
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.neighbors import (
    KNeighborsClassifier,
    KNeighborsRegressor
)

from sklearn.metrics import (
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

### Classification Model

In [14]:
# Download the Breast Cancer dataset
path = kagglehub.dataset_download(
    "uciml/breast-cancer-wisconsin-data"
)

# Load the dataset
df = pd.read_csv(f"{path}/data.csv")
print(df.head())

# Drop unnecessary columns
df.drop(columns=["id", "Unnamed: 32"], inplace=True)

# Separate features and target
X = df.drop(columns="diagnosis")
y = df["diagnosis"]

# Encode the target labels
y = y.map({"M": 1, "B": 0})

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Create the preprocessing pipeline
preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Create the model pipeline
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier())
    ]
)

# Define the hyperparameter search space
param_grid = {
    "model__n_neighbors": np.arange(1, 31),
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
    "model__p": [1, 2]
}

# Stratified Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Perform Randomized Search
search = RandomizedSearchCV(
    pipeline,
    param_grid,
    n_iter=20,
    scoring="f1",
    cv=cv,
    random_state=42,
    n_jobs=-1
)

# Train the classifier
search.fit(X_train, y_train)

print(search.best_params_)

         id diagnosis  radius_mean  texture_mean  perimeter_mean  area_mean  \
0    842302         M        17.99         10.38          122.80     1001.0   
1    842517         M        20.57         17.77          132.90     1326.0   
2  84300903         M        19.69         21.25          130.00     1203.0   
3  84348301         M        11.42         20.38           77.58      386.1   
4  84358402         M        20.29         14.34          135.10     1297.0   

   smoothness_mean  compactness_mean  concavity_mean  concave points_mean  \
0          0.11840           0.27760          0.3001              0.14710   
1          0.08474           0.07864          0.0869              0.07017   
2          0.10960           0.15990          0.1974              0.12790   
3          0.14250           0.28390          0.2414              0.10520   
4          0.10030           0.13280          0.1980              0.10430   

   ...  texture_worst  perimeter_worst  area_worst  smoothness

{'model__weights': 'distance', 'model__p': 1, 'model__n_neighbors': np.int64(3), 'model__metric': 'euclidean'}


In [15]:
# Classification Report
classifier = search.best_estimator_

predictions = classifier.predict(X_test)

print("Classification Report\n")
print(classification_report(y_test, predictions))

Classification Report

              precision    recall  f1-score   support

           0       0.92      0.99      0.95        72
           1       0.97      0.86      0.91        42

    accuracy                           0.94       114
   macro avg       0.95      0.92      0.93       114
weighted avg       0.94      0.94      0.94       114



### Regression Model

In [16]:
# Download California Housing dataset
path = kagglehub.dataset_download(
    "camnugent/california-housing-prices"
)

# Load the dataset
df = pd.read_csv(f"{path}/housing.csv")

print(df.head())

# Separate features and target
X = df.drop(columns="median_house_value")
y = df["median_house_value"]

# Identify numerical and categorical features
numerical_features = X.select_dtypes(exclude="object").columns
categorical_features = X.select_dtypes(include="object").columns

# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create the model pipeline
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", KNeighborsRegressor())
    ]
)

# Define the hyperparameter search space
param_grid = {
    "model__n_neighbors": np.arange(1, 31),
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan"],
    "model__p": [1, 2]
}

# K-Fold Cross Validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Perform Randomized Search
search = RandomizedSearchCV(
    pipeline,
    param_grid,
    n_iter=20,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=42,
    n_jobs=-1
)

# Train the regressor
search.fit(X_train, y_train)

print(search.best_params_)

   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1      2401.0      1138.0         8.3014            358500.0        NEAR BAY  
2       496.0       177.0         7.2574            352100.0        NEAR BAY  
3       558.0       219.0         5.6431            341300.0        NEAR BAY  
4       565.0       259.0         3.8462            342200.0        NEAR BAY  


{'model__weights': 'distance', 'model__p': 1, 'model__n_neighbors': np.int64(17), 'model__metric': 'manhattan'}


In [19]:
# Regression Metrics
regressor = search.best_estimator_

predictions = regressor.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

n = len(y_test)
p = regressor.named_steps["preprocessor"].transform(X_train.iloc[:1]).shape[1]

print(f"MAE         : {mae:.2f}")
print(f"RMSE        : {rmse:.2f}")


MAE         : 38986.57
RMSE        : 58252.91


# 3. Parameters and Hyperparameters

Unlike many machine learning algorithms, K-Nearest Neighbors does not estimate any model parameters during training. The algorithm simply stores the training data and performs all computations during prediction.

---

## Model Parameters

KNN has no learned parameters. \
Unlike Linear Regression, Decision Trees or Neural Networks, KNN does not estimate coefficients, decision rules or weights. \
The training phase simply stores the feature vectors and corresponding target values.

---

## Hyperparameters

### 1. n_neighbors
Default Value: 5 \
Admissible Values: Positive integer, typically between 1 and 30 \
Determines the number of nearest neighbors used to make a prediction. 

| Small K | Large K |
|----------|----------|
| Complex model | Simpler model |
| Low bias | High bias |
| High variance | Low variance |
| May overfit | May underfit |
| Sensitive to noise | More stable |

### 2. weights
Default Value: uniform \
Available Values: uniform, distance \
Determines how neighbors contribute to the prediction.
- uniform: Every neighbor contributes equally.
- distance: Closer neighbors receive higher weights (preferred).


### 3. metric
Default Value: minkowski \
Common Values: minkowski, euclidean, manhattan, chebyshev, cosine, hamming \
Determines how distances between observations are computed.
- Euclidean distance for continuous numerical features.
- Manhattan distance when robustness to outliers is desired.
- Cosine distance for text or high-dimensional sparse data.

### 4. p
Default Value: 2 \
Available Values: Positive integer \
Used only when metric='minkowski'.


### 5. algorithm
Default Value: auto \
Available Values: auto, ball_tree, kd_tree, brute \
Specifies the algorithm used to search for nearest neighbors.

| Algorithm | Best For |
|------------|-----------|
| auto | Automatically selects the most suitable algorithm |
| kd_tree | Low-dimensional numerical data |
| ball_tree | Higher-dimensional numerical data |
| brute | Small datasets or unsupported distance metrics |

### 6. leaf_size
Default Value: 30
Controls the size of leaf nodes in KDTree and BallTree.
- Smaller values increase tree depth.
- Larger values reduce tree depth.
- Usually does not require tuning unless working with very large datasets.

### 7. n_jobs
Default Value: None
Available Values: None, -1, Positive Integer
Controls the number of CPU cores used during neighbor searches.


### Parameter Interactions

- Small values of `n_neighbors` generally benefit from `distance` weighting.
- Euclidean distance (`p=2`) requires feature scaling.
- The effectiveness of `kd_tree` decreases as the number of features increases.
- Larger datasets benefit from optimized neighbor search algorithms instead of brute-force search.

### Key Takeaways

- KNN has no learned model parameters.
- The choice of `n_neighbors` has the greatest impact on model performance.
- Distance metric and feature scaling strongly influence prediction quality.
- Hyperparameter tuning should always be performed using cross-validation.

# 4. Data Requirements

Since KNN is a distance-based algorithm, the quality of the input data has a significant impact on its performance. Proper preprocessing is often more important than the choice of the algorithm itself.

### Feature Scaling
KNN computes distances between observations. Features with larger numerical ranges dominate the distance calculation, causing features with smaller ranges to have little influence. \
Always scale numerical features before training a KNN model.

### Outliers
Outliers can become the nearest neighbors of new observations, leading to poor predictions. \
Investigate and treat outliers before fitting the model.

### Missing Values
Use an imputation strategy within a preprocessing pipeline.

### Categorical Features
Distance metrics require numerical inputs.

### High-Dimensional Data
As dimensionality increases, distances between observations become increasingly similar. This phenomenon is known as the Curse of Dimensionality.

### Class Imbalance (Classification)
Majority classes tend to dominate the nearest neighbors. Hence, essential to treat this.

### Feature Selection
Irrelevant features contribute to distance calculations and may reduce prediction accuracy. \
Remove irrelevant or redundant features whenever possible.

### Multicollinearity
KNN is a distance-based algorithm and does not estimate model coefficients. Therefore, multicollinearity does not cause unstable parameter estimates as it does in linear models. \
However, highly correlated features may contribute redundant information to the distance calculation, causing certain characteristics to be overrepresented and potentially degrading prediction performance. \
Although KNN is not severely affected by multicollinearity, removing redundant features may improve prediction accuracy and computational efficiency, particularly in high-dimensional datasets.

# 5. Summary

### Advantages

- Simple and easy to understand.
- Supports both classification and regression.
- No training phase is required.
- Naturally models complex, non-linear decision boundaries.
- No assumptions about the underlying data distribution.
- Easily adapts to newly added training samples.

### Limitations

- Prediction is computationally expensive.
- Requires feature scaling.
- Highly sensitive to outliers and noisy observations.
- Performance deteriorates in high-dimensional datasets.
- Memory intensive because the entire training dataset must be stored.
- Selecting an appropriate value of K is crucial.

### When to Use

- Small to medium-sized datasets.
- Low-dimensional feature spaces.
- Non-linear decision boundaries.
- Problems where similar observations are expected to have similar outcomes.

### When Not to Use

- Very large datasets.
- High-dimensional data.
- Real-time prediction systems requiring extremely fast inference.
- Datasets containing many irrelevant features or significant noise.